# Notebook 6 — Evaluation (Validation + Final Test)

**Goal:** Evaluate association-rule recommendations learned from `train_rules` (Notebook 5)
on two different sets:
- **validation**: the last PRIOR order kept as validation
- **test (final)**: Instacart TRAIN order (the next order of each user)

We evaluate a "basket completion" task:
- For each basket, we hide 1 item (ground truth)
- We use the remaining items as input
- We generate top-K recommendations using the association rules
- We compare predictions to the hidden item

Metrics:
- **precision@K** = (#hits in top-K) / K
- **recall@K** = (#hits in top-K) / (#hidden items)
- **hit-rate@K** = 1 if at least 1 hit, else 0
- **coverage@K** = (#unique recommended items across baskets) / (#candidate items)

In [4]:
import os
import ast
import random
import numpy as np
import pandas as pd
from IPython.display import display

In [5]:
# Paths
OUTPUT_DIR = "../outputs"
DATA_DIR = "../data"

In [6]:
def load_baskets_outputs():
    """
    Load basket files created in Notebook 4 from the outputs folder.

    Returns
    -------
    dict
        Basket tables (all, train_rules, validation, test).
    """
    tables = {
        "baskets_all": pd.read_csv(f"{OUTPUT_DIR}/baskets_all.csv"),
        "baskets_train_rules": pd.read_csv(f"{OUTPUT_DIR}/baskets_train_rules.csv"),
        "baskets_validation": pd.read_csv(f"{OUTPUT_DIR}/baskets_validation.csv"),
        "baskets_test": pd.read_csv(f"{OUTPUT_DIR}/baskets_test.csv"),
    }
    return tables


def parse_items_column(baskets_df):
    """
    Convert the 'items' column from string to Python list.

    Returns
    -------
    DataFrame
        Same table with parsed 'items' column.
    """
    df = baskets_df.copy()
    df["items"] = df["items"].apply(ast.literal_eval)
    return df


def load_rules_csv(prefix="apriori_train_rules"):
    """
    Load association rules saved in Notebook 5.

    Expected file:
    - {prefix}_rules.csv

    Returns
    -------
    DataFrame
        Rules table.
    """
    rules = pd.read_csv(f"{OUTPUT_DIR}/{prefix}_rules.csv")
    return rules


def load_selected_products(prefix="apriori_train_rules"):
    """
    Load selected products (the reduced product space) used in Notebook 5.

    Expected file:
    - {prefix}_selected_products.csv

    Returns
    -------
    DataFrame
        Selected products table.
    """
    df = pd.read_csv(f"{OUTPUT_DIR}/{prefix}_selected_products.csv")
    return df


# %%
def parse_rule_itemset_string(s):
    """
    Parse a saved rule itemset string like "13176|47209" into a set of ints.

    Returns
    -------
    set
        Set of product_id integers.
    """
    if pd.isna(s) or str(s).strip() == "":
        return set()
    parts = str(s).split("|")
    out = set()
    for p in parts:
        p = p.strip()
        if p != "":
            out.add(int(p))
    return out


def prepare_rules_for_recommendation(rules_df, min_lift=1.0):
    """
    Prepare rules for fast recommendation.

    We keep only rules with:
    - single-item consequent (subset -> item)
    - lift > min_lift

    We build a list of rules:
    (antecedent_set, consequent_item, score, confidence, lift, support)

    Score choice (simple):
    - score = confidence * lift

    Returns
    -------
    list
        Prepared rule list.
    """
    df = rules_df.copy()

    # If Notebook 5 saved as strings "a|b|c", we parse those columns
    # (If your file still contains python tuples, you must keep the save format from Notebook 5.)
    if "antecedents" in df.columns and df["antecedents"].dtype == object:
        # We expect "a|b|c" format from Notebook 5 save function
        pass

    # Ensure single consequent
    # Here we rely on the saved "consequents" string being single item like "13176"
    df["ante_set"] = df["antecedents"].apply(parse_rule_itemset_string)
    df["con_set"] = df["consequents"].apply(parse_rule_itemset_string)

    df["antecedent_len"] = df["ante_set"].apply(len)
    df["consequent_len"] = df["con_set"].apply(len)

    df = df[(df["consequent_len"] == 1)].copy()

    # Filter by lift
    if "lift" in df.columns:
        df = df[df["lift"] > min_lift].copy()

    prepared = []
    for _, row in df.iterrows():
        ante = row["ante_set"]
        con_item = list(row["con_set"])[0]
        conf = float(row["confidence"]) if "confidence" in row else 0.0
        lift = float(row["lift"]) if "lift" in row else 1.0
        supp = float(row["support"]) if "support" in row else 0.0

        score = conf * lift
        prepared.append((frozenset(ante), int(con_item), score, conf, lift, supp))

    return prepared


def recommend_top_k(observed_items, rules_prepared, top_k=10):
    """
    Generate top-K recommendations using prepared rules.

    For each rule:
    - if antecedent is subset of observed_items, it can recommend its consequent
    - we aggregate by taking the best score per recommended item

    Parameters
    ----------
    observed_items : set
        Items available as input basket.
    rules_prepared : list
        Output of prepare_rules_for_recommendation().
    top_k : int
        Number of recommendations.

    Returns
    -------
    list
        List of recommended product_ids.
    """
    obs = set(observed_items)
    best_score = {}

    for ante, con, score, conf, lift, supp in rules_prepared:
        if ante.issubset(obs):
            # Keep the best score if multiple rules recommend same item
            if (con not in best_score) or (score > best_score[con]):
                best_score[con] = score

    if len(best_score) == 0:
        return []

    # Sort by score desc
    ranked = sorted(best_score.items(), key=lambda x: x[1], reverse=True)
    recs = [pid for pid, sc in ranked[:top_k]]
    return recs


# %%
def hide_items_for_evaluation(items, n_hide=1, seed=42):
    """
    Hide n items from a basket for evaluation (simple random hide).

    Returns
    -------
    tuple
        (observed_list, hidden_list)
    """
    rng = random.Random(seed)

    if len(items) <= n_hide:
        return items, []

    items_list = list(items)
    rng.shuffle(items_list)

    hidden = items_list[:n_hide]
    observed = items_list[n_hide:]

    return observed, hidden


def evaluate_baskets(
    baskets_df,
    rules_prepared,
    candidate_items_set,
    top_k=10,
    n_hide=1,
    seed=42,
    max_baskets=None
):
    """
    Evaluate recommendations on a basket dataset.

    For each basket:
    - hide n items (ground truth)
    - predict top-K from observed items
    - compute per-basket precision/recall/hit

    Then aggregate:
    - mean precision@K
    - mean recall@K
    - hit-rate@K (mean of hits)
    - coverage@K: unique recommended items / number of candidate items

    Parameters
    ----------
    baskets_df : DataFrame
        Must contain 'items' as list.
    rules_prepared : list
        Prepared rule list.
    candidate_items_set : set
        The product universe used for mining (selected products).
    top_k : int
        K for top-K recommendations.
    n_hide : int
        Number of hidden items per basket.
    seed : int
        Random seed.
    max_baskets : int or None
        If not None, evaluate only first max_baskets baskets.

    Returns
    -------
    tuple
        (summary_df, per_basket_df)
    """
    rows = []
    unique_recommended = set()

    n_total = 0
    n_skipped = 0

    # To make the hiding reproducible but still different per basket,
    # we change seed with basket index
    for i, row in baskets_df.iterrows():
        if max_baskets is not None and n_total >= max_baskets:
            break

        items = row["items"]

        # Basic guard: need at least 2 items to hide 1 and still recommend
        if not isinstance(items, list) or len(items) < 2:
            n_skipped += 1
            continue

        # Keep only candidate items (same product space as mining)
        items = [p for p in items if p in candidate_items_set]
        if len(items) < 2:
            n_skipped += 1
            continue

        observed, hidden = hide_items_for_evaluation(items, n_hide=n_hide, seed=seed + i)

        if len(hidden) == 0 or len(observed) == 0:
            n_skipped += 1
            continue

        recs = recommend_top_k(set(observed), rules_prepared, top_k=top_k)

        # Update coverage
        for r in recs:
            unique_recommended.add(r)

        hidden_set = set(hidden)
        recs_set = set(recs)

        hits = len(hidden_set.intersection(recs_set))

        precision = hits / top_k if top_k > 0 else 0.0
        recall = hits / len(hidden_set) if len(hidden_set) > 0 else 0.0
        hit_rate = 1.0 if hits > 0 else 0.0

        rows.append({
            "order_id": row["order_id"] if "order_id" in baskets_df.columns else np.nan,
            "user_id": row["user_id"] if "user_id" in baskets_df.columns else np.nan,
            "segment": row["segment"] if "segment" in baskets_df.columns else "unknown",
            "basket_size": len(items),
            "observed_size": len(observed),
            "hidden_size": len(hidden),
            "top_k": top_k,
            "n_recs": len(recs),
            "hits": hits,
            "precision_at_k": precision,
            "recall_at_k": recall,
            "hit_rate_at_k": hit_rate
        })

        n_total += 1

    per_basket = pd.DataFrame(rows)

    # Aggregate
    if len(per_basket) == 0:
        summary = pd.DataFrame([{
            "n_evaluated_baskets": 0,
            "n_skipped_baskets": n_skipped,
            "top_k": top_k,
            "n_hide": n_hide,
            "mean_precision_at_k": np.nan,
            "mean_recall_at_k": np.nan,
            "hit_rate_at_k": np.nan,
            "coverage_at_k": np.nan
        }])
        return summary, per_basket

    coverage = len(unique_recommended) / max(1, len(candidate_items_set))

    summary = pd.DataFrame([{
        "n_evaluated_baskets": int(len(per_basket)),
        "n_skipped_baskets": int(n_skipped),
        "top_k": int(top_k),
        "n_hide": int(n_hide),
        "mean_precision_at_k": float(per_basket["precision_at_k"].mean()),
        "mean_recall_at_k": float(per_basket["recall_at_k"].mean()),
        "hit_rate_at_k": float(per_basket["hit_rate_at_k"].mean()),
        "coverage_at_k": float(coverage)
    }])

    return summary, per_basket


def evaluate_by_segment(per_basket_df):
    """
    Evaluate metrics separately by customer segment.

    Returns
    -------
    DataFrame
        Segment-level summary.
    """
    if len(per_basket_df) == 0:
        return pd.DataFrame()

    seg = per_basket_df.groupby("segment").agg(
        n_baskets=("hits", "size"),
        mean_precision_at_k=("precision_at_k", "mean"),
        mean_recall_at_k=("recall_at_k", "mean"),
        hit_rate_at_k=("hit_rate_at_k", "mean")
    ).reset_index()

    return seg


def save_evaluation_outputs(prefix, summary_df, per_basket_df, seg_df):
    """
    Save evaluation outputs to OUTPUT_DIR.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    summary_df.to_csv(f"{OUTPUT_DIR}/{prefix}_summary.csv", index=False)
    per_basket_df.to_csv(f"{OUTPUT_DIR}/{prefix}_per_basket.csv", index=False)
    seg_df.to_csv(f"{OUTPUT_DIR}/{prefix}_by_segment.csv", index=False)

In [7]:
# 1) Load data (baskets + rules + selected products)
basket_tables = load_baskets_outputs()

baskets_validation = parse_items_column(basket_tables["baskets_validation"])
baskets_test = parse_items_column(basket_tables["baskets_test"])

rules_df = load_rules_csv(prefix="apriori_train_rules")
selected_products_df = load_selected_products(prefix="apriori_train_rules")

candidate_items_set = set(selected_products_df["product_id"].astype(int).tolist())

display(pd.DataFrame([{
    "n_validation_baskets": len(baskets_validation),
    "n_test_baskets": len(baskets_test),
    "n_selected_products_candidate_space": len(candidate_items_set),
    "n_rules_loaded": len(rules_df)
}]))

,n_validation_baskets,n_test_baskets,n_selected_products_candidate_space,n_rules_loaded
0,206209,131209,3000,203


In [8]:
# 2) Prepare rules for recommendation
# Keep rules with lift > 1 (clean) and single-item consequent
rules_prepared = prepare_rules_for_recommendation(rules_df, min_lift=1.0)

display(pd.DataFrame([{
    "n_rules_prepared_for_reco": len(rules_prepared)
}]))

,n_rules_prepared_for_reco
0,203


In [9]:
# 3) Evaluate on VALIDATION
TOP_K = 10
N_HIDE = 1
SEED = 42

val_summary, val_per_basket = evaluate_baskets(
    baskets_validation,
    rules_prepared,
    candidate_items_set=candidate_items_set,
    top_k=TOP_K,
    n_hide=N_HIDE,
    seed=SEED,
    max_baskets=None
)

val_by_segment = evaluate_by_segment(val_per_basket)

display(val_summary)
display(val_by_segment)

save_evaluation_outputs("eval_validation_apriori_train_rules", val_summary, val_per_basket, val_by_segment)

,n_evaluated_baskets,n_skipped_baskets,top_k,n_hide,mean_precision_at_k,mean_recall_at_k,hit_rate_at_k,coverage_at_k
0,179095,27114,10,1,0.005878,0.058779,0.058779,0.006333


,segment,n_baskets,mean_precision_at_k,mean_recall_at_k,hit_rate_at_k
0,frequent,61945,0.005661,0.056615,0.056615
1,heavy,64849,0.007218,0.072183,0.072183
2,rare,52301,0.004472,0.044722,0.044722


In [10]:
# 4) Evaluate on FINAL TEST (Instacart TRAIN order)
test_summary, test_per_basket = evaluate_baskets(
    baskets_test,
    rules_prepared,
    candidate_items_set=candidate_items_set,
    top_k=TOP_K,
    n_hide=N_HIDE,
    seed=SEED,
    max_baskets=None
)

test_by_segment = evaluate_by_segment(test_per_basket)

display(test_summary)
display(test_by_segment)

save_evaluation_outputs("eval_test_apriori_train_rules", test_summary, test_per_basket, test_by_segment)

,n_evaluated_baskets,n_skipped_baskets,top_k,n_hide,mean_precision_at_k,mean_recall_at_k,hit_rate_at_k,coverage_at_k
0,114031,17178,10,1,0.005912,0.059124,0.059124,0.006333


,segment,n_baskets,mean_precision_at_k,mean_recall_at_k,hit_rate_at_k
0,frequent,38972,0.005809,0.058093,0.058093
1,heavy,41282,0.007129,0.071290,0.071290
2,rare,33777,0.004545,0.045445,0.045445


In [11]:
# 5) Optional: Compare VALIDATION vs TEST in one table
comparison = pd.concat([
    val_summary.assign(dataset="validation"),
    test_summary.assign(dataset="test_final")
], ignore_index=True)

display(comparison)

comparison.to_csv(f"{OUTPUT_DIR}/eval_compare_validation_vs_test.csv", index=False)

,n_evaluated_baskets,n_skipped_baskets,top_k,n_hide,mean_precision_at_k,mean_recall_at_k,hit_rate_at_k,coverage_at_k,dataset
0,179095,27114,10,1,0.005878,0.058779,0.058779,0.006333,validation
1,114031,17178,10,1,0.005912,0.059124,0.059124,0.006333,test_final
